# Part 3: Event Detection Using Vision-Language Models

This notebook performs VLM-based event detection for the Group 3 TVSUM videos:
- TVSUM 9
- TVSUM 10
- TVSUM 11
- TVSUM 12

The notebook:
1. Samples frames from each video.
2. Uses a Vision-Language Model to detect salient events.
3. Runs two prompting conditions:
   - Event-only detection
   - Event + timestamp detection
4. Parses the VLM outputs into structured tables.

Expected folder structure:

```text
Comp Vision/
    vlm_event_detection_group3.ipynb
    videos/
        video_9.mp4
        video_10.mp4
        video_11.mp4
        video_12.mp4
```

In [ ]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

In [ ]:
# Run this cell once inside your new cv_vlm_env kernel.
!python -m pip install --upgrade pip
!pip install opencv-python pillow pandas numpy matplotlib tqdm
!pip install transformers accelerate qwen-vl-utils

In [ ]:
import os
import re
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from tqdm import tqdm

In [ ]:
# =========================
# CONFIGURATION
# =========================

GROUP_ID = "Group3"
DATASET_NAME = "TVSUM"

# Folder where your videos are stored.
# Since your notebook is inside:
# C:\Users\aggel\OneDrive\Υπολογιστής\Comp Vision
# this relative path points to:
# C:\Users\aggel\OneDrive\Υπολογιστής\Comp Vision\videos
VIDEO_DIR = Path("videos")

# Folder where outputs will be saved
OUTPUT_DIR = Path("outputs_vlm_event_detection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Group 3 videos: your exact filenames
VIDEOS = {
    "TVSUM_9": VIDEO_DIR / "video_9.mp4",
    "TVSUM_10": VIDEO_DIR / "video_10.mp4",
    "TVSUM_11": VIDEO_DIR / "video_11.mp4",
    "TVSUM_12": VIDEO_DIR / "video_12.mp4",
}

# Sample one frame every N seconds
FRAME_EVERY_N_SECONDS = 2

# Maximum number of frames sent to the VLM.
# Keep this small first to avoid memory problems.
MAX_FRAMES_FOR_VLM = 12

In [ ]:
for video_name, video_path in VIDEOS.items():
    print(video_name, "->", video_path)

    if video_path.exists():
        print("  Found")
    else:
        print("  NOT FOUND")

In [ ]:
def seconds_to_mmss(seconds):
    """
    Convert seconds to MM:SS format.

    Example:
    75 -> 01:15
    """
    seconds = int(round(seconds))
    minutes = seconds // 60
    secs = seconds % 60
    return f"{minutes:02d}:{secs:02d}"


def mmss_to_seconds(time_str):
    """
    Convert MM:SS format to seconds.

    Example:
    01:15 -> 75
    """
    minutes, seconds = time_str.strip().split(":")
    return int(minutes) * 60 + int(seconds)

In [ ]:
def sample_frames_from_video(video_path, output_frame_dir, every_n_seconds=2, max_frames=None):
    """
    Sample frames from a video every N seconds.

    Parameters
    ----------
    video_path:
        Path to the input video.

    output_frame_dir:
        Folder where sampled frames will be saved.

    every_n_seconds:
        Sampling interval in seconds.

    max_frames:
        Maximum number of frames to keep.
        If the video produces more frames, we uniformly select max_frames.

    Returns
    -------
    sampled_frames:
        A list of dictionaries containing timestamp, seconds, and frame path.
    """

    video_path = Path(video_path)
    output_frame_dir = Path(output_frame_dir)
    output_frame_dir.mkdir(parents=True, exist_ok=True)

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    if fps == 0:
        raise ValueError(f"Could not read FPS from video: {video_path}")

    duration_seconds = frame_count / fps

    sampled_frames = []

    timestamps = np.arange(0, duration_seconds, every_n_seconds)

    # If too many frames, select max_frames uniformly across the whole video
    if max_frames is not None and len(timestamps) > max_frames:
        indices = np.linspace(0, len(timestamps) - 1, max_frames).astype(int)
        timestamps = timestamps[indices]

    for t in timestamps:
        frame_index = int(t * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)

        success, frame = cap.read()

        if not success:
            continue

        # OpenCV reads in BGR, but PIL/matplotlib use RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        timestamp = seconds_to_mmss(t)
        frame_filename = f"frame_{timestamp.replace(':', '-')}.jpg"
        frame_path = output_frame_dir / frame_filename

        Image.fromarray(frame_rgb).save(frame_path)

        sampled_frames.append({
            "timestamp": timestamp,
            "seconds": int(t),
            "frame_path": str(frame_path)
        })

    cap.release()

    return sampled_frames

In [ ]:
# Test frame sampling on TVSUM 9

video_name = "TVSUM_9"
video_path = VIDEOS[video_name]

frame_dir = OUTPUT_DIR / video_name / "sampled_frames"

sampled_frames = sample_frames_from_video(
    video_path=video_path,
    output_frame_dir=frame_dir,
    every_n_seconds=FRAME_EVERY_N_SECONDS,
    max_frames=MAX_FRAMES_FOR_VLM
)

sampled_frames_df = pd.DataFrame(sampled_frames)
display(sampled_frames_df)

In [ ]:
def display_sampled_frames(sampled_frames, cols=4):
    """
    Display sampled frames with timestamps.
    """

    n = len(sampled_frames)

    if n == 0:
        print("No frames to display.")
        return

    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(4 * cols, 3 * rows))

    for i, item in enumerate(sampled_frames):
        image = Image.open(item["frame_path"])

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(item["timestamp"])
        plt.axis("off")

    plt.tight_layout()
    plt.show()


display_sampled_frames(sampled_frames)

In [ ]:
EVENT_ONLY_PROMPT = """
You are given sampled frames from a video in chronological order.

Task:
Retrieve all of the salient events of the video.

Rules:
- List only important events needed to understand the video.
- Keep the events in chronological order.
- Use short and clear descriptions.
- Do not describe every frame.
- Do not include timestamps.
- Do not include unimportant visual details.
- Use this exact output format:

Salient event 1: <event description>
Salient event 2: <event description>
Salient event 3: <event description>
"""


EVENT_TIMESTAMP_PROMPT = """
You are given sampled frames from a video in chronological order.
Each frame is labeled with its timestamp.

Task:
Retrieve all of the salient events of the video with temporal localization.

Rules:
- List only important events needed to understand the video.
- Keep events in chronological order.
- Use short and clear descriptions.
- Use precise but realistic timestamps.
- Use MM:SS - MM:SS format.
- Do not describe every frame.
- Do not include unimportant visual details.
- Use this exact output format:

Salient event 1: <event description>, <MM:SS - MM:SS>
Salient event 2: <event description>, <MM:SS - MM:SS>
Salient event 3: <event description>, <MM:SS - MM:SS>
"""

print(EVENT_ONLY_PROMPT)
print(EVENT_TIMESTAMP_PROMPT)

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

if device == "cpu":
    model = model.to(device)

In [ ]:
def build_vlm_message(sampled_frames, prompt):
    """
    Build a Qwen2.5-VL message containing sampled frames and a text prompt.
    """

    content = []

    for item in sampled_frames:
        content.append({
            "type": "text",
            "text": f"Frame from {item['timestamp']}:"
        })

        content.append({
            "type": "image",
            "image": item["frame_path"]
        })

    content.append({
        "type": "text",
        "text": prompt
    })

    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    return messages

In [ ]:
def run_qwen_vlm(sampled_frames, prompt, max_new_tokens=512):
    """
    Run Qwen2.5-VL on sampled frames using the given prompt.
    """

    messages = build_vlm_message(sampled_frames, prompt)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text

In [ ]:
# Run event-only prompt on TVSUM 9

event_only_output = run_qwen_vlm(
    sampled_frames=sampled_frames,
    prompt=EVENT_ONLY_PROMPT,
    max_new_tokens=512
)

print(event_only_output)

In [ ]:
# Run event + timestamp prompt on TVSUM 9

event_timestamp_output = run_qwen_vlm(
    sampled_frames=sampled_frames,
    prompt=EVENT_TIMESTAMP_PROMPT,
    max_new_tokens=512
)

print(event_timestamp_output)

In [ ]:
def save_text(text, path):
    """
    Save text to a file.
    """

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


video_output_dir = OUTPUT_DIR / video_name
video_output_dir.mkdir(parents=True, exist_ok=True)

save_text(event_only_output, video_output_dir / "vlm_event_only_raw.txt")
save_text(event_timestamp_output, video_output_dir / "vlm_event_timestamp_raw.txt")

In [ ]:
def parse_event_only(vlm_text):
    """
    Parse event-only VLM output.

    Expected format:
    Salient event 1: description
    Salient event 2: description

    Returns:
    DataFrame with columns:
    - event_id
    - event_description
    """

    pattern = r"Salient event\s*(\d+)\s*:\s*(.+)"
    rows = []

    for match in re.finditer(pattern, vlm_text, flags=re.IGNORECASE):
        event_id = int(match.group(1))
        description = match.group(2).strip()

        # Remove accidental timestamp if model included one
        description = re.sub(
            r",?\s*\d{2}:\d{2}\s*-\s*\d{2}:\d{2}",
            "",
            description
        ).strip()

        rows.append({
            "event_id": event_id,
            "event_description": description
        })

    return pd.DataFrame(rows)

In [ ]:
def parse_event_with_timestamps(vlm_text):
    """
    Parse VLM output with timestamps.

    Expected format:
    Salient event 1: description, 00:03 - 00:07

    Returns:
    DataFrame with columns:
    - event_id
    - event_description
    - start
    - end
    - start_seconds
    - end_seconds
    """

    pattern = r"Salient event\s*(\d+)\s*:\s*(.+?),\s*(\d{2}:\d{2})\s*-\s*(\d{2}:\d{2})"
    rows = []

    for match in re.finditer(pattern, vlm_text, flags=re.IGNORECASE):
        event_id = int(match.group(1))
        description = match.group(2).strip()
        start = match.group(3).strip()
        end = match.group(4).strip()

        rows.append({
            "event_id": event_id,
            "event_description": description,
            "start": start,
            "end": end,
            "start_seconds": mmss_to_seconds(start),
            "end_seconds": mmss_to_seconds(end)
        })

    return pd.DataFrame(rows)

In [ ]:
parsed_event_only_df = parse_event_only(event_only_output)
parsed_timestamp_df = parse_event_with_timestamps(event_timestamp_output)

print("Event-only parsed output:")
display(parsed_event_only_df)

print("Event + timestamp parsed output:")
display(parsed_timestamp_df)

In [ ]:
parsed_event_only_df.to_csv(
    video_output_dir / "parsed_event_only.csv",
    index=False
)

parsed_timestamp_df.to_csv(
    video_output_dir / "parsed_event_timestamp.csv",
    index=False
)

In [ ]:
def process_video_until_parsing(video_name, video_path):
    """
    Full pipeline until parsed VLM outputs:
    1. sample frames
    2. run event-only prompt
    3. run event+timestamp prompt
    4. parse outputs
    5. save results
    """

    print(f"\n==============================")
    print(f"Processing {video_name}")
    print(f"==============================")

    video_output_dir = OUTPUT_DIR / video_name
    frame_dir = video_output_dir / "sampled_frames"

    video_output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Sample frames
    sampled_frames = sample_frames_from_video(
        video_path=video_path,
        output_frame_dir=frame_dir,
        every_n_seconds=FRAME_EVERY_N_SECONDS,
        max_frames=MAX_FRAMES_FOR_VLM
    )

    sampled_frames_df = pd.DataFrame(sampled_frames)

    sampled_frames_df.to_csv(
        video_output_dir / "sampled_frames_metadata.csv",
        index=False
    )

    print(f"Sampled {len(sampled_frames)} frames.")

    # 2. Run event-only prompt
    print("Running event-only prompt...")
    event_only_output = run_qwen_vlm(
        sampled_frames=sampled_frames,
        prompt=EVENT_ONLY_PROMPT,
        max_new_tokens=512
    )

    save_text(event_only_output, video_output_dir / "vlm_event_only_raw.txt")

    # 3. Run event + timestamp prompt
    print("Running event + timestamp prompt...")
    event_timestamp_output = run_qwen_vlm(
        sampled_frames=sampled_frames,
        prompt=EVENT_TIMESTAMP_PROMPT,
        max_new_tokens=512
    )

    save_text(event_timestamp_output, video_output_dir / "vlm_event_timestamp_raw.txt")

    # 4. Parse outputs
    parsed_event_only_df = parse_event_only(event_only_output)
    parsed_timestamp_df = parse_event_with_timestamps(event_timestamp_output)

    parsed_event_only_df.to_csv(
        video_output_dir / "parsed_event_only.csv",
        index=False
    )

    parsed_timestamp_df.to_csv(
        video_output_dir / "parsed_event_timestamp.csv",
        index=False
    )

    print("\nEvent-only parsed output:")
    display(parsed_event_only_df)

    print("\nEvent + timestamp parsed output:")
    display(parsed_timestamp_df)

    return {
        "video_name": video_name,
        "sampled_frames": sampled_frames_df,
        "event_only_output": event_only_output,
        "event_timestamp_output": event_timestamp_output,
        "parsed_event_only": parsed_event_only_df,
        "parsed_timestamp": parsed_timestamp_df
    }

In [ ]:
# Run the whole pipeline for all Group 3 videos.
# This may take a while because it runs the VLM twice per video.

all_parsed_results = {}

for video_name, video_path in VIDEOS.items():
    result = process_video_until_parsing(video_name, video_path)
    all_parsed_results[video_name] = result

In [ ]:
# Combine all parsed event-only outputs into one CSV

all_event_only_dfs = []

for video_name, result in all_parsed_results.items():
    df = result["parsed_event_only"].copy()
    df.insert(0, "video_name", video_name)
    all_event_only_dfs.append(df)

all_event_only_df = pd.concat(all_event_only_dfs, ignore_index=True)

display(all_event_only_df)

all_event_only_df.to_csv(
    OUTPUT_DIR / "all_videos_parsed_event_only.csv",
    index=False
)

In [ ]:
# Combine all parsed timestamp outputs into one CSV

all_timestamp_dfs = []

for video_name, result in all_parsed_results.items():
    df = result["parsed_timestamp"].copy()
    df.insert(0, "video_name", video_name)
    all_timestamp_dfs.append(df)

all_timestamp_df = pd.concat(all_timestamp_dfs, ignore_index=True)

display(all_timestamp_df)

all_timestamp_df.to_csv(
    OUTPUT_DIR / "all_videos_parsed_event_timestamp.csv",
    index=False
)

## Outputs

After running the notebook, you should have:

```text
outputs_vlm_event_detection/
    TVSUM_9/
        sampled_frames/
        sampled_frames_metadata.csv
        vlm_event_only_raw.txt
        vlm_event_timestamp_raw.txt
        parsed_event_only.csv
        parsed_event_timestamp.csv

    TVSUM_10/
        ...

    TVSUM_11/
        ...

    TVSUM_12/
        ...

    all_videos_parsed_event_only.csv
    all_videos_parsed_event_timestamp.csv
```

This finishes the first version of Part 3 until parsing.